In [1]:
# COMMAND: Install API packages

!pip install -q flask pyngrok pandas joblib scikit-learn

print("API packages installed successfully!")

API packages installed successfully!


In [2]:
# COMMAND: Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully!")

Mounted at /content/drive
Google Drive mounted successfully!


In [3]:
# COMMAND: Import required libraries

import os
import joblib
import pandas as pd

from flask import Flask, request, jsonify

print("All API libraries imported successfully!")

All API libraries imported successfully!


In [4]:
# COMMAND: Define project and model paths

project_path = (
    "/content/drive/MyDrive/"
    "Alumni_Donor_Propensity_Forecaster"
)

model_path = os.path.join(
    project_path,
    "models",
    "alumni_donor_model_pipeline.pkl"
)

print("Project Path:")
print(project_path)

print("\nModel Path:")
print(model_path)

Project Path:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster

Model Path:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/models/alumni_donor_model_pipeline.pkl


In [5]:
# COMMAND: Verify that the trained model exists

import os

if os.path.exists(model_path):

    print("SUCCESS!")
    print("Trained model found.")

else:

    print("ERROR!")
    print("Trained model was not found.")

print("\nModel:")
print(model_path)

SUCCESS!
Trained model found.

Model:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/models/alumni_donor_model_pipeline.pkl


In [6]:
# COMMAND: Load the pretrained ML pipeline

import joblib

model = joblib.load(
    model_path
)

print("Pretrained model loaded successfully!")

print("\nModel type:")
print(type(model))

Pretrained model loaded successfully!

Model type:
<class 'sklearn.pipeline.Pipeline'>


In [7]:
# COMMAND: Create Flask API application

from flask import Flask

app = Flask(
    __name__
)

print("Flask application created successfully!")

Flask application created successfully!


In [8]:
# COMMAND: Create API home endpoint

@app.route(
    "/",
    methods=["GET"]
)
def home():

    return jsonify({
        "project":
            "Alumni Donor Propensity Forecaster",

        "status":
            "API is running",

        "endpoint":
            "/predict",

        "method":
            "POST"
    })


print("Home endpoint created successfully!")

Home endpoint created successfully!


In [9]:
# COMMAND: Create /predict REST API endpoint

@app.route(
    "/predict",
    methods=["POST"]
)
def predict():

    try:

        # Receive JSON data
        data = request.get_json()

        # Validate JSON
        if data is None:

            return jsonify({
                "error":
                    "Request body must contain JSON data."
            }), 400

        # Convert JSON into DataFrame
        input_data = pd.DataFrame(
            [data]
        )

        # Generate prediction
        prediction = model.predict(
            input_data
        )[0]

        # Generate probability
        probability = model.predict_proba(
            input_data
        )[0][1]

        # Convert probability to percentage
        propensity_score = (
            probability * 100
        )

        # Create category
        if propensity_score >= 70:

            category = "High"

            interpretation = (
                "Likely to Donate"
            )

        elif propensity_score >= 40:

            category = "Medium"

            interpretation = (
                "Moderate Donation Propensity"
            )

        else:

            category = "Low"

            interpretation = (
                "Less Likely to Donate"
            )

        # Convert numerical prediction
        prediction_label = (
            "Yes"
            if prediction == 1
            else "No"
        )

        # Return response
        return jsonify({

            "prediction":
                prediction_label,

            "donation_probability":
                round(probability, 4),

            "propensity_score":
                round(propensity_score, 2),

            "category":
                category,

            "interpretation":
                interpretation

        })

    except Exception as e:

        return jsonify({

            "error":
                str(e)

        }), 400


print("Prediction endpoint created successfully!")

Prediction endpoint created successfully!


In [10]:
# COMMAND: Create API information endpoint

@app.route(
    "/api-info",
    methods=["GET"]
)
def api_info():

    return jsonify({

        "project":
            "Alumni Donor Propensity Forecaster",

        "model":
            "Random Forest Classifier",

        "target":
            "Is_Donor_2025",

        "prediction_values":
            ["Yes", "No"],

        "propensity_categories":
            ["High", "Medium", "Low"],

        "prediction_endpoint":
            "/predict",

        "prediction_method":
            "POST"

    })


print("API information endpoint created!")

API information endpoint created!


In [11]:
# COMMAND: Start Flask server locally

app.run(
    host="0.0.0.0",
    port=8000,
    debug=False
)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
INFO:werkzeug:Press CTRL+C to quit


In [12]:
# COMMAND: Run Flask server in background

import threading

def run_flask():

    app.run(
        host="0.0.0.0",
        port=8000,
        debug=False,
        use_reloader=False
    )


flask_thread = threading.Thread(
    target=run_flask
)

flask_thread.daemon = True

flask_thread.start()

print("Flask server started in background!")
print("Local API: http://127.0.0.1:8000")

Flask server started in background!
Local API: http://127.0.0.1:8000
 * Serving Flask app '__main__'
 * Debug mode: off


In [13]:
# COMMAND: Test Flask API locally

import requests

response = requests.get(
    "http://127.0.0.1:8000/"
)

print("Status Code:")
print(response.status_code)

print("\nResponse:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [23/Sep/2026 05:11:23] "GET / HTTP/1.1" 200 -


Status Code:
200

Response:
{'endpoint': '/predict', 'method': 'POST', 'project': 'Alumni Donor Propensity Forecaster', 'status': 'API is running'}


In [14]:
# COMMAND: Test API information endpoint

import requests

response = requests.get(
    "http://127.0.0.1:8000/api-info"
)

print("Status Code:")
print(response.status_code)

print("\nAPI Information:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [23/Sep/2026 05:11:34] "GET /api-info HTTP/1.1" 200 -


Status Code:
200

API Information:
{'model': 'Random Forest Classifier', 'prediction_endpoint': '/predict', 'prediction_method': 'POST', 'prediction_values': ['Yes', 'No'], 'project': 'Alumni Donor Propensity Forecaster', 'propensity_categories': ['High', 'Medium', 'Low'], 'target': 'Is_Donor_2025'}


In [15]:
# COMMAND: Create sample alumni JSON request

sample_alumni = {

    "Age": 45,

    "Gender": "Male",

    "Graduation_Year": 2005,

    "Degree_Level": "Master",

    "Major": "Business",

    "Alumni_Status": "Active",

    "Email_Available": "Yes",

    "Phone_Available": "Yes",

    "Wealth_Rating": "A",

    "Income_Bracket": "$100K-$150K",

    "Event_Attendance": 8,

    "Email_Open_Rate": 75.5,

    "Location": "New York, USA",

    "Consecutive_Giving_Years": 5,

    "Donation_Last_Year_2024": 1500.00,

    "Total_Lifetime_Giving": 10000.00
}

print("Sample alumni JSON created!")

print(sample_alumni)

Sample alumni JSON created!
{'Age': 45, 'Gender': 'Male', 'Graduation_Year': 2005, 'Degree_Level': 'Master', 'Major': 'Business', 'Alumni_Status': 'Active', 'Email_Available': 'Yes', 'Phone_Available': 'Yes', 'Wealth_Rating': 'A', 'Income_Bracket': '$100K-$150K', 'Event_Attendance': 8, 'Email_Open_Rate': 75.5, 'Location': 'New York, USA', 'Consecutive_Giving_Years': 5, 'Donation_Last_Year_2024': 1500.0, 'Total_Lifetime_Giving': 10000.0}


In [16]:
# COMMAND: Send alumni data to prediction API

import requests

api_url = (
    "http://127.0.0.1:8000/predict"
)

response = requests.post(
    api_url,
    json=sample_alumni
)

print("Status Code:")
print(response.status_code)

print("\nAPI Response:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [23/Sep/2026 05:12:03] "POST /predict HTTP/1.1" 200 -


Status Code:
200

API Response:
{'category': 'High', 'donation_probability': 0.86, 'interpretation': 'Likely to Donate', 'prediction': 'Yes', 'propensity_score': 86.0}


In [17]:
# COMMAND: Display prediction response

response_data = response.json()

print("=" * 60)
print("       ALUMNI DONOR PROPENSITY RESULT")
print("=" * 60)

print(
    "Prediction:",
    response_data.get("prediction")
)

print(
    "Donation Probability:",
    response_data.get(
        "donation_probability"
    )
)

print(
    "Propensity Score:",
    response_data.get(
        "propensity_score"
    )
)

print(
    "Category:",
    response_data.get(
        "category"
    )
)

print(
    "Interpretation:",
    response_data.get(
        "interpretation"
    )
)

print("=" * 60)

       ALUMNI DONOR PROPENSITY RESULT
Prediction: Yes
Donation Probability: 0.86
Propensity Score: 86.0
Category: High
Interpretation: Likely to Donate


In [18]:
# COMMAND: Install ngrok Python package

!pip install -q pyngrok

print("pyngrok installed successfully!")

pyngrok installed successfully!


In [19]:
# COMMAND: Configure ngrok authentication

from pyngrok import ngrok

NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_AUTH_TOKEN_HERE"

ngrok.set_auth_token(
    NGROK_AUTH_TOKEN
)

print("ngrok authentication configured!")

ngrok authentication configured!


In [22]:
# COMMAND: Configure ngrok authentication

!pip install -q pyngrok

from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3JdKuZGguEKbHkt38xe4Eq7an1m_216wean7ywaN1wVtxJJT7"
NGROK_AUTH_TOKEN = ""

ngrok.set_auth_token(
    NGROK_AUTH_TOKEN
)

print("ngrok authentication configured successfully!")

ngrok authentication configured successfully!


In [24]:
# COMMAND: Start Flask API in background

import threading

def run_flask():
    app.run(
        host="0.0.0.0",
        port=8000,
        debug=False,
        use_reloader=False
    )

flask_thread = threading.Thread(
    target=run_flask
)

flask_thread.daemon = True
flask_thread.start()

print("Flask API started successfully!")
print("Local API: http://127.0.0.1:8000")

Flask API started successfully!
Local API: http://127.0.0.1:8000
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 8000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [25]:
# COMMAND: Test local Flask API

import requests

response = requests.get(
    "http://127.0.0.1:8000/"
)

print("Status Code:", response.status_code)

print("\nResponse:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [23/Sep/2026 05:16:56] "GET / HTTP/1.1" 200 -


Status Code: 200

Response:
{'endpoint': '/predict', 'method': 'POST', 'project': 'Alumni Donor Propensity Forecaster', 'status': 'API is running'}


In [27]:
# COMMAND: Install or update pyngrok

!pip install -q --upgrade pyngrok

print("pyngrok installed successfully!")

pyngrok installed successfully!


In [28]:
# COMMAND: Configure ngrok authentication

from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3JdKuZGguEKbHkt38xe4Eq7an1m_216wean7ywaN1wVtxJJT7"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("ngrok authentication configured successfully!")

ngrok authentication configured successfully!


In [29]:
# COMMAND: Verify ngrok authentication

from pyngrok import ngrok

try:
    tunnels = ngrok.get_tunnels()
    print("ngrok authentication is configured.")
    print("Existing tunnels:", tunnels)

except Exception as e:
    print("ngrok verification failed:")
    print(e)

ngrok authentication is configured.
Existing tunnels: []


In [30]:
# COMMAND: Start Flask API

import threading

def run_flask():
    app.run(
        host="0.0.0.0",
        port=8000,
        debug=False,
        use_reloader=False
    )

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

print("Flask API started successfully!")
print("Local API: http://127.0.0.1:8000")

 * Serving Flask app '__main__'
Flask API started successfully!
Local API: http://127.0.0.1:8000
 * Debug mode: off


Address already in use
Port 8000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [31]:
# COMMAND: Test local Flask API

import requests

response = requests.get("http://127.0.0.1:8000/")

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [23/Sep/2026 05:18:24] "GET / HTTP/1.1" 200 -


Status Code: 200
Response: {'endpoint': '/predict', 'method': 'POST', 'project': 'Alumni Donor Propensity Forecaster', 'status': 'API is running'}


In [32]:
# COMMAND: Create public ngrok tunnel

from pyngrok import ngrok

# Close old ngrok sessions
ngrok.kill()

# Create new public tunnel
public_url = ngrok.connect(8000, "http")

print("=" * 60)
print("NGROK TUNNEL CREATED SUCCESSFULLY")
print("=" * 60)

print("\nPublic API URL:")
print(public_url)

print("\nPrediction Endpoint:")
print(str(public_url) + "/predict")

NGROK TUNNEL CREATED SUCCESSFULLY

Public API URL:
NgrokTunnel: "https://stank-epidermis-phonebook.ngrok-free.dev" -> "http://localhost:8000"

Prediction Endpoint:
NgrokTunnel: "https://stank-epidermis-phonebook.ngrok-free.dev" -> "http://localhost:8000"/predict


In [33]:
# COMMAND: Get clean public API URL

public_api_url = public_url.public_url

print("=" * 60)
print("PUBLIC API URL")
print("=" * 60)

print(public_api_url)

print("\nPrediction Endpoint:")
print(public_api_url + "/predict")

PUBLIC API URL
https://stank-epidermis-phonebook.ngrok-free.dev

Prediction Endpoint:
https://stank-epidermis-phonebook.ngrok-free.dev/predict


In [34]:
# COMMAND: Test public Flask API

import requests

response = requests.get(public_api_url)

print("Status Code:", response.status_code)

print("\nAPI Response:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [23/Sep/2026 05:20:25] "GET / HTTP/1.1" 200 -


Status Code: 200

API Response:
{'endpoint': '/predict', 'method': 'POST', 'project': 'Alumni Donor Propensity Forecaster', 'status': 'API is running'}


In [35]:
# COMMAND: Create sample alumni input for API testing

sample_alumni = {
    "Age": 45,
    "Gender": "Male",
    "Graduation_Year": 2005,
    "Degree_Level": "Master",
    "Major": "Business",
    "Alumni_Status": "Active",
    "Email_Available": "Yes",
    "Phone_Available": "Yes",
    "Wealth_Rating": "A",
    "Income_Bracket": "$100K-$150K",
    "Event_Attendance": 8,
    "Email_Open_Rate": 75.5,
    "Location": "New York, USA",
    "Consecutive_Giving_Years": 5,
    "Donation_Last_Year_2024": 1500.00,
    "Total_Lifetime_Giving": 10000.00
}

print("Sample alumni data created successfully!")

print("\nInput Data:")
print(sample_alumni)

Sample alumni data created successfully!

Input Data:
{'Age': 45, 'Gender': 'Male', 'Graduation_Year': 2005, 'Degree_Level': 'Master', 'Major': 'Business', 'Alumni_Status': 'Active', 'Email_Available': 'Yes', 'Phone_Available': 'Yes', 'Wealth_Rating': 'A', 'Income_Bracket': '$100K-$150K', 'Event_Attendance': 8, 'Email_Open_Rate': 75.5, 'Location': 'New York, USA', 'Consecutive_Giving_Years': 5, 'Donation_Last_Year_2024': 1500.0, 'Total_Lifetime_Giving': 10000.0}


In [36]:
# COMMAND: Send alumni data to public prediction API

import requests

predict_url = public_api_url + "/predict"

response = requests.post(
    predict_url,
    json=sample_alumni
)

print("Status Code:", response.status_code)

print("\nAPI Response:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [23/Sep/2026 05:20:41] "POST /predict HTTP/1.1" 200 -


Status Code: 200

API Response:
{'category': 'High', 'donation_probability': 0.86, 'interpretation': 'Likely to Donate', 'prediction': 'Yes', 'propensity_score': 86.0}


In [37]:
# COMMAND: Display final donor propensity result

result = response.json()

print("=" * 65)
print("          ALUMNI DONOR PROPENSITY RESULT")
print("=" * 65)

print(f"Prediction           : {result.get('prediction')}")
print(f"Donation Probability : {result.get('donation_probability')}")
print(f"Propensity Score     : {result.get('propensity_score')}%")
print(f"Category             : {result.get('category')}")
print(f"Interpretation       : {result.get('interpretation')}")

print("=" * 65)

          ALUMNI DONOR PROPENSITY RESULT
Prediction           : Yes
Donation Probability : 0.86
Propensity Score     : 86.0%
Category             : High
Interpretation       : Likely to Donate


In [38]:
# COMMAND: Save API prediction response to Google Drive

import json
import os

api_result_path = os.path.join(
    project_path,
    "results",
    "api_prediction_response.json"
)

with open(api_result_path, "w") as file:
    json.dump(
        result,
        file,
        indent=4
    )

print("API result saved successfully!")

print("\nFile location:")
print(api_result_path)

API result saved successfully!

File location:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/results/api_prediction_response.json


In [39]:
# COMMAND: Verify Phase 4 REST API completion

print("=" * 70)
print("       ALUMNI DONOR PROPENSITY FORECASTER")
print("                    PHASE 4")
print("=" * 70)

print("\n✓ Dataset loaded")
print("✓ Random Forest model loaded")
print("✓ Flask API running")
print("✓ ngrok authentication successful")
print("✓ Public tunnel created")
print("✓ GET / working")
print("✓ POST /predict working")
print("✓ Prediction generated")
print("✓ API response saved")

print("\nPublic API URL:")
print(public_api_url)

print("\nPrediction Endpoint:")
print(public_api_url + "/predict")

print("\nPrediction Result:")
print(result)

print("\n" + "=" * 70)
print("             PHASE 4 COMPLETED!")
print("=" * 70)

       ALUMNI DONOR PROPENSITY FORECASTER
                    PHASE 4

✓ Dataset loaded
✓ Random Forest model loaded
✓ Flask API running
✓ ngrok authentication successful
✓ Public tunnel created
✓ GET / working
✓ POST /predict working
✓ Prediction generated
✓ API response saved

Public API URL:
https://stank-epidermis-phonebook.ngrok-free.dev

Prediction Endpoint:
https://stank-epidermis-phonebook.ngrok-free.dev/predict

Prediction Result:
{'category': 'High', 'donation_probability': 0.86, 'interpretation': 'Likely to Donate', 'prediction': 'Yes', 'propensity_score': 86.0}

             PHASE 4 COMPLETED!
